In [1]:
from pathlib import Path

In [28]:
data_folder = Path(
    "/dh-projects/ag-ishaque/raw_data/tiesmeys-ovrlpy/Xenium-brain-2024/replicate1"
)

transcript_file = data_folder / "transcripts.csv.gz"

# Run Baysor

The Xenium brain parquet has the problem that thee gene column is encoded as binary w/o the specification of String, therefore the gene filtering does not seem to work, also baysor does not support csv.gz

In [31]:
import os

In [32]:
partition = "-p compute-96cpu-700GB-RAM"
time = 5 * 24  # h
mem = 650  # GB
threads = 8

In [41]:
csv = "transcripts.csv"
unzip_cmd = f"gunzip -c {transcript_file.resolve()} > {csv}"
rm_cmd = f"rm {csv}"

baysor_cmd = (
    f"JULIA_NUM_THREADS={threads} ~/.julia/bin/baysor run {csv} :cell_id -c xenium.toml"
)

baysor_cmd = " && ".join([unzip_cmd, baysor_cmd, rm_cmd])

In [43]:
cmd = (
    f"sbatch -J baysor --mem={mem}G -n {threads} -N 1 "
    "-o baysor_log.txt "
    f"--time={time}:00:00 "
    f"{partition} "
    f"--wrap='{baysor_cmd}'"
)
cmd

"sbatch -J baysor --mem=650G -n 8 -N 1 -o baysor_log.txt --time=120:00:00 -p compute-96cpu-700GB-RAM --wrap='gunzip -c /dh-projects/ag-ishaque/raw_data/tiesmeys-ovrlpy/Xenium-brain-2024/replicate1/transcripts.csv.gz > transcripts.csv && JULIA_NUM_THREADS=8 ~/.julia/bin/baysor run transcripts.csv :cell_id -c xenium.toml && rm transcripts.csv'"

In [44]:
os.popen(cmd).read()

'Submitted batch job 4380790\n'